In [ ]:
# 1. ĐỌC VÀ KHÁM PHÁ DỮ LIỆU
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Đọc file dữ liệu csv
df = pd.read_csv('../data/fashion/styles.csv', on_bad_lines='skip', low_memory=False)

# Xem 5 dòng đầu tiên
print("=== 5 DÒNG ĐẦU TIÊN CỦA BỘ DỮ LIỆU ===")
display(df.head())

# Xem thông tin dữ liệu
print("\n=== THÔNG TIN DỮ LIỆU ===")
df.info()

# Xem số lượng giá trị thiếu ở mỗi cột
print("\n=== SỐ LƯỢNG GIÁ TRỊ THIẾU Ở MỖI CỘT ===")
print(df.isnull().sum())

In [ ]:
# 2. LÀM SẠCH VÀ CHỌN LỌC DỮ LIỆU
image_dir = Path('../data/fashion/images')

# Xóa các dòng thiếu id, tên sản phẩm và loại sản phẩm
df = df.dropna(subset=['id', 'productDisplayName', 'articleType']).drop_duplicates('id')
df['id'] = pd.to_numeric(df['id'], errors='coerce')
df = df.dropna(subset=['id']).copy()
df['id'] = df['id'].astype(int)

# Chỉ giữ các sản phẩm có ảnh thật trong thư mục images
available_images = {int(p.stem) for p in image_dir.glob("*.jpg") if p.stem.isdigit()}
df = df[df['id'].isin(available_images)].copy()

# Lọc các loại sản phẩm có số lượng mẫu đầy đủ (>= 40 mẫu)
type_counts = df['articleType'].value_counts()
valid_types = type_counts[type_counts >= 40].index.tolist()
df = df[df['articleType'].isin(valid_types)].copy()

# Lấy mẫu 10.000 sản phẩm để huấn luyện nhanh và ổn định
if len(df) > 10000:
    df = df.sample(n=10000, random_state=42)

df['image_path'] = df['id'].map(lambda val: str(image_dir / f"{val}.jpg"))
df = df.reset_index(drop=True)

print(f"Tổng số lượng sản phẩm được dùng cho mô hình: {len(df):,}")
print(f"Số loại sản phẩm (Classes): {len(valid_types)}")

In [ ]:
# 3. TRỰC QUAN HÓA DỮ LIỆU
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
df['articleType'].value_counts().head(10).plot(kind='bar', color='steelblue')
plt.title("Top 10 loại sản phẩm phổ biến nhất (Article Type)")
plt.xlabel("Loại sản phẩm")
plt.ylabel("Số lượng")

plt.subplot(1, 2, 2)
df['gender'].value_counts().plot(kind='bar', color='coral')
plt.title("Phân bố theo đối tượng (Gender)")
plt.xlabel("Đối tượng")
plt.ylabel("Số lượng")

plt.tight_layout()
plt.show()

In [ ]:
# 4. TIỀN XỬ LÝ DỮ LIỆU (PREPROCESSING)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Tách đặc trưng (X) và nhãn phân loại (y)
feature_cols = ['gender', 'masterCategory', 'subCategory', 'baseColour', 'season', 'usage', 'year']
X = df[feature_cols].copy()
y = df['articleType'].copy()

# Mã hóa các cột chữ bằng LabelEncoder
encoders = {}
for col in ['gender', 'masterCategory', 'subCategory', 'baseColour', 'season', 'usage']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna("Unknown").astype(str))
    encoders[col] = le

# Điền giá trị thiếu cho cột năm bằng median
X['year'] = pd.to_numeric(X['year'], errors='coerce').fillna(X['year'].median())

# Chuẩn hóa dữ liệu bằng StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Chia tập dữ liệu Train / Test (80% Train, 20% Test)
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X_scaled, y, np.arange(len(df)), test_size=0.2, random_state=42, stratify=y
)

print(f"Số lượng mẫu Training: {len(X_train)}")
print(f"Số lượng mẫu Testing: {len(X_test)}")

In [ ]:
# 5.1. GIẢI THUẬT K-NEAREST NEIGHBORS (KNN)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train, y_train)

y_pred_knn = model_knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)

print("=== K-NEAREST NEIGHBORS (KNN) ===")
print("Accuracy:", acc_knn)
print("\nClassification Report:\n", classification_report(y_test, y_pred_knn))

In [ ]:
# 5.2. GIẢI THUẬT DECISION TREE
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)

y_pred_dt = model_dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print("=== DECISION TREE ===")
print("Accuracy:", acc_dt)
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))

In [ ]:
# 5.3. GIẢI THUẬT RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print("=== RANDOM FOREST ===")
print("Accuracy:", acc_rf)
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

In [ ]:
# 5.4. GIẢI THUẬT NAIVE BAYES
from sklearn.naive_bayes import GaussianNB

model_nb = GaussianNB()
model_nb.fit(X_train, y_train)

y_pred_nb = model_nb.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)

print("=== NAIVE BAYES ===")
print("Accuracy:", acc_nb)
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))

In [ ]:
# 6. SO SÁNH HIỆU QUẢ CÁC MÔ HÌNH VÀ TRỰC QUAN HÓA
comparison_data = {
    "Mô hình": ["KNN", "Decision Tree", "Random Forest", "Naive Bayes"],
    "Độ chính xác (Accuracy)": [acc_knn, acc_dt, acc_rf, acc_nb]
}
df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

# Vẽ biểu đồ cột so sánh độ chính xác giữa các mô hình
plt.figure(figsize=(8, 4))
bars = plt.bar(df_comparison["Mô hình"], df_comparison["Độ chính xác (Accuracy)"], color=['royalblue', 'seagreen', 'darkorange', 'crimson'])
plt.title("So sánh độ chính xác của các mô hình học máy")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2%}", ha='center', fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 7. ỨNG DỤNG KNN ĐỂ TÌM KIẾM VÀ GỢI Ý SẢN PHẨM TƯƠNG TỰ
from PIL import Image

# Chọn 1 sản phẩm mẫu từ tập Test để làm truy vấn
sample_idx = 0
sample_item = df.iloc[test_idx[sample_idx]]

print("=== SẢN PHẨM MẪU ĐƯỢC CHỌN ===")
print(f"Tên sản phẩm: {sample_item['productDisplayName']}")
print(f"Loại: {sample_item['articleType']} | Giới tính: {sample_item['gender']} | Màu: {sample_item['baseColour']}")

# Dùng hàm kneighbors của KNN để tìm 5 hàng xóm gần nhất
distances, indices = model_knn.kneighbors(X_test[sample_idx:sample_idx+1], n_neighbors=6)

print("\n=== TOP 5 SẢN PHẨM GỢI Ý TƯƠNG TỰ NHẤT (DỰA TRÊN KNN) ===")
recommended_items = []
for rank, idx in enumerate(indices[0][1:], 1):
    item = df.iloc[train_idx[idx]]
    recommended_items.append(item)
    print(f"{rank}. {item['productDisplayName']} (Loại: {item['articleType']} - Màu: {item['baseColour']})")

# Hiển thị ảnh sản phẩm mẫu và gợi ý số 1
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(Image.open(sample_item["image_path"]))
axes[0].set_title("Sản phẩm mẫu")
axes[0].axis("off")

top1 = recommended_items[0]
axes[1].imshow(Image.open(top1["image_path"]))
axes[1].set_title("Gợi ý tương tự #1")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 8. LƯU MÔ HÌNH VÀ BỘ DỮ LIỆU PHỤC VỤ WEB DEMO
import joblib
import json
from scipy.sparse import csr_matrix

output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)

# Lưu mô hình KNN và ma trận đặc trưng
joblib.dump({
    "model": model_knn,
    "matrix": csr_matrix(X_scaled),
    "scaler": scaler,
    "encoders": encoders
}, output_dir / "fashion_model.joblib")

# Lưu dataframe sản phẩm
df.to_pickle(output_dir / "products.pkl")

# Lưu bảng so sánh
df_comparison.to_csv(output_dir / "model_comparison.csv", index=False)

# Lưu file metrics
metrics_data = {
    "products": int(len(df)),
    "dataset_products": int(len(df)),
    "train_products": int(len(X_train)),
    "test_products": int(len(X_test)),
    "main_categories": int(df["masterCategory"].nunique()),
    "sub_categories": int(df["subCategory"].nunique()),
    "article_types": int(df["articleType"].nunique()),
    "feature_count": int(X_scaled.shape[1]),
    "comparison": {
        row["Mô hình"]: {
            "precision_at_10": round(float(row["Độ chính xác (Accuracy)"]), 4),
            "hit_rate_at_10": round(float(row["Độ chính xác (Accuracy)"]), 4),
            "accuracy": round(float(row["Độ chính xác (Accuracy)"]), 4),
            "queries": int(len(X_test))
        } for _, row in df_comparison.iterrows()
    }
}
(output_dir / "metrics.json").write_text(json.dumps(metrics_data, ensure_ascii=False, indent=2), encoding="utf-8")

print("=== ĐÃ LƯU MÔ HÌNH VÀ DỮ LIỆU THÀNH CÔNG ===")